# Graph-Paper AI — Colab Test

Vectorless Graph-RAG: PDF → Graph → Tree Search (LLM) → Answer.

Works with or without a local Ollama instance.

## Setup

In [ ]:
# Clone the repo and install deps
import sys, subprocess, os

REPO_URL = "https://github.com/anomalyco/graph-paper-ai.git"
REPO_DIR = "/content/graph-paper-ai"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q pymupdf networkx httpx

In [ ]:
from pathlib import Path
from pprint import pprint

import networkx as nx

from src.ingestion import (
    ImageInfo, ProcessingResult,
    build_adjacency_map, build_page_index,
    build_section_tree, detect_co_located_blocks,
    extract_cross_references, parse_paper, print_tree,
)
from src.llm.ollama_client import OllamaClient, OllamaMessage
from src.retrieval.tree_search import TreeSearchResult, answer_query, tree_search

## 1. Get a PDF

Run this cell, then click the **Upload** button to select a PDF from your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
PDF_PATH = Path(pdf_name)
OUTPUT_DIR = Path("output")
print(f"Using: {PDF_PATH} ({PDF_PATH.stat().st_size / 1024:.0f} KB)")

### Or download a sample paper automatically

In [ ]:
# Uncomment to skip upload and use a sample arXiv paper instead:
# PDF_URL = "https://arxiv.org/pdf/2401.12345.pdf"  # Change this
# !wget -q -O paper.pdf {PDF_URL}
# PDF_PATH = Path("paper.pdf")
# OUTPUT_DIR = Path("output")
# print(f"Downloaded: {PDF_PATH}")

## 2. Parse the PDF

Extracts markdown (with `## Page N` markers), images, and spatial co-location edges.

In [ ]:
result: ProcessingResult = parse_paper(PDF_PATH, output_dir=OUTPUT_DIR)

print(f"Source:      {result.metadata['source']}")
print(f"Pages:       {result.metadata['page_count']}")
print(f"Images:      {len(result.images)}")
print(f"Spatial edges: {len(result.edges)}")
print(f"Markdown:    {len(result.markdown):,} chars")
print()
print("─ Markdown preview (first 600 chars) ─")
print(result.markdown[:600])

## 3. Build the Graph

Section hierarchy + figure/formula nodes + cross-references + spatial edges.

In [ ]:
refs = extract_cross_references(result.markdown)
graph: nx.DiGraph = build_adjacency_map(result, refs=refs)

print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

type_counts = {}
for _, attr in graph.nodes(data=True):
    nt = attr.get("node_type", "unknown")
    type_counts[nt] = type_counts.get(nt, 0) + 1
pprint(type_counts)

## 4. Section Tree (LLM search target)

The LLM sees this tree with `node_id`s and returns the ones relevant to the query.

In [ ]:
section_nodes = build_section_tree(graph)
tree_str = print_tree(section_nodes, show_ids=True)
print(tree_str)

## 5. Test with Mock LLM (no Ollama required)

Simulates tree search + answer synthesis. Replace with real Ollama below.

In [ ]:
from unittest.mock import MagicMock

QUERY = "What is the main contribution?"  # ← Change this

mock_llm = MagicMock()

# Mock step 1: LLM picks first 3 sections from the tree
if section_nodes:
    mock_ids = ",".join(n.node_id for n in section_nodes[:3])
    mock_llm.chat.return_value = mock_ids

    search_result = tree_search(QUERY, graph, mock_llm)
    print(f"LLM selected: {search_result.selected_ids}")
    print(f"Context: {len(search_result.context):,} chars")
    print()
    print("─ Context preview ─")
    print(search_result.context[:1500])
    print()

    # Mock step 2: answer
    mock_llm.chat.return_value = (
        f"Based on the sections [{', '.join(search_result.selected_ids[:3])}], "
        f"the main contribution is clearly explained in the extracted context."
    )
    answer = answer_query(QUERY, search_result.context, mock_llm)
    print("🧠 Answer:")
    print(answer)
else:
    print("No sections in the tree. The PDF may lack heading structure.")

## 6. (Optional) Query with Real Ollama

If you have Ollama running and accessible (e.g., via ngrok or local network), configure below.

Otherwise skip — the mock test above already validates the pipeline.

In [ ]:
# ── Uncomment and configure if you have Ollama accessible ──

# OLLAMA_URL = "http://YOUR_SERVER_IP:11434"  # e.g. ngrok URL or local IP
# OLLAMA_MODEL = "llama3"

# llm = OllamaClient(base_url=OLLAMA_URL, model=OLLAMA_MODEL)

# if llm.is_available():
#     print("✅ Ollama connected")
#
#     QUERY = "Summarize the key findings of this paper."
#
#     search_result = tree_search(QUERY, graph, llm)
#     print(f"Selected: {search_result.selected_ids}")
#     print(f"Context: {len(search_result.context):,} chars")
#     print()
#
#     answer = answer_query(QUERY, search_result.context, llm)
#     print("🧠 Answer:")
#     print(answer)
# else:
#     print("❌ Ollama not reachable")